# Compsognathus: learn quiet bilateral balance

[Open in Colab](https://colab.research.google.com/github/kuds/mesozoic-labs/blob/codex/compsognathus-foot-research/notebooks/compsognathus_balance_study.ipynb)

Choose **Runtime → Run all** for one work session. The notebook first checks saved progress and shows completed runs, remaining work, and the next job. The default `session` mode prepares any missing references, home calibration and training probes, then works on **one remaining 11M-step training job**, prioritizing an unfinished job. Its default soft budget is **14 hours**; set `SESSION_HOURS` to 12 for a shorter session. If the budget expires, it saves a PPO checkpoint and pauses. Use the same study name and Run all in your next Colab session to continue automatically.

Source loads directly from PR527; no source ZIP or model upload is needed. The initial checks read existing Drive reports, replay the compatible saved Compy model over 40 episodes, calibrate 40 home-controller episodes, and run four 8,192-step probes. Completed work is validated and reused. Existing experiments stay unchanged, while new results go to a separate study folder. A = current reward/filter off; B = current reward/10 Hz; C = bilateral reward/filter off; D = bilateral reward/10 Hz. Each full arm uses seeds 42, 43 and 44. Historical checkpoints remain comparison baselines, and the new training arms initially start fresh.

The complete study still contains **132,055,040 training steps** across four probes and 12 full runs. The previous Compy 11M run took **13h 32m 53s**, making one job a useful session target. Twelve runs at that historical pace take **162h 34m 36s — about 163 hours (6.8 days)** before additional checks. That run used a Colab L4 runtime; this study uses CPU training and adds physics-rate evaluations, so speed may differ. The 14-hour budget is a checkpoint-based stopping target, not a guarantee that a job will finish or the runtime will stop at exactly that time. The session budget includes startup progress inspection, references, calibration, probes and training. Installing the runtime and the final Drive flush add time; a current update, evaluation or checkpoint save can also finish beyond the soft deadline. Abrupt runtime loss can prevent the final flush, so checkpoints are also saved throughout training.


In [ ]:
# @title Choose the work session
import math

RUN_MODE = "session"  # @param ["session", "full", "smoke"]
SESSION_HOURS = 14  # @param {type:"number"}
STUDY_NAME = "compsognathus-balance-v1"  # @param {type:"string"}
COMPY_REFERENCE_RUN = "20260909_162812"  # @param {type:"string"}
TREX_REFERENCE_RUN = "20260821_142144"  # @param {type:"string"}

# session: missing prerequisites, then one remaining full job with a soft time budget.
# full: all remaining jobs without the session time cap.
# smoke: references, home calibration and the four short probes only.
if RUN_MODE not in {"session", "full", "smoke"}:
    raise ValueError("RUN_MODE must be session, full or smoke")
if RUN_MODE == "session" and (
    isinstance(SESSION_HOURS, bool)
    or not isinstance(SESSION_HOURS, (int, float))
    or not math.isfinite(SESSION_HOURS)
    or SESSION_HOURS <= 0
):
    raise ValueError("SESSION_HOURS must be a finite positive number")

In [ ]:
# @title Mount existing Drive and load source directly from the PR
import json
import os
import re
import subprocess
import sys
import uuid
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive did not mount; persistent storage is required")
BASE = Path("/content/drive/MyDrive/mesozoic-labs")
if not STUDY_NAME or STUDY_NAME in {".", ".."} or Path(STUDY_NAME).name != STUDY_NAME:
    raise ValueError("STUDY_NAME must be one folder name")
STUDY = BASE / STUDY_NAME
STUDY.mkdir(parents=True, exist_ok=True)
token = str(uuid.uuid4())
storage_check = STUDY / (".write-check-" + token)
try:
    with storage_check.open("w") as stream:
        stream.write(token)
        stream.flush()
        os.fsync(stream.fileno())
    if storage_check.read_text() != token:
        raise IOError("Drive write verification failed")
finally:
    storage_check.unlink(missing_ok=True)

EXPECTED_IMPLEMENTATION = "sha256:57333732cdad0406208022d068bcfad9b8b6c616ee5cc588759b6d145b45bb1e"
saved_plan = STUDY / "study_plan.json"
saved = json.loads(saved_plan.read_text()) if saved_plan.exists() else None
revision = saved["source_commit"] if saved else "refs/pull/527/head"
if saved and (not re.fullmatch(r"[0-9a-f]{40}", revision) or saved["implementation_sha256"] != EXPECTED_IMPLEMENTATION):
    raise RuntimeError("Open the notebook snapshot saved with this study, or use a new study name for changed source")

REPO = Path("/content/mesozoic-balance-pr527")
REMOTE = "https://github.com/kuds/mesozoic-labs.git"


def git(*args):
    return subprocess.check_output(["git", "-C", str(REPO), *args], text=True).strip()


if not (REPO / ".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError("Source directory contains other files; use a fresh runtime")
    REPO.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "init", "-q", str(REPO)])
    git("remote", "add", "origin", REMOTE)
if git("remote", "get-url", "origin") != REMOTE:
    raise RuntimeError("Source checkout has an unexpected remote")
if git("status", "--porcelain"):
    raise RuntimeError("Source checkout contains changes; use a fresh runtime")
git("fetch", "--quiet", "--depth=1", "origin", revision)
target = git("rev-parse", "FETCH_HEAD")
if any(name == "environments" or name.startswith("environments.") for name in sys.modules):
    if git("rev-parse", "HEAD") != target:
        raise RuntimeError("Study modules are already imported from another revision; use a fresh runtime")
git("checkout", "--quiet", "--detach", target)
os.chdir(REPO)
print("Source revision:", target)
print("Existing experiments:", BASE / "logs")
print("New persistent results:", STUDY)

In [ ]:
# @title Install the training dependencies
import subprocess
import sys

# On a later session, restore the recorded core package versions before import.
# Python itself is supplied by Colab; a changed Python needs a matching runtime.
install = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[train]"]
saved_plan = STUDY / "study_plan.json"
if saved_plan.exists():
    runtime = json.loads(saved_plan.read_text())["runtime"]
    if sys.version.split()[0] != runtime["python"]:
        raise RuntimeError(
            "This study requires Python "
            + runtime["python"]
            + "; this runtime has "
            + sys.version.split()[0]
            + ". Use a Colab runtime with the recorded Python version. Do not mix results across runtimes."
        )
    constraints = Path("/content/balance-study-constraints.txt")
    constraints.write_text("\n".join(name + "==" + value for name, value in runtime.items() if name != "python") + "\n")
    install.extend(["--constraint", str(constraints)])
subprocess.check_call(install)
sys.path.insert(0, str(REPO))
from environments.compsognathus.experiments.balance_identity import study_source_fingerprint

if study_source_fingerprint() != EXPECTED_IMPLEMENTATION:
    raise ValueError("Installed study source does not match this notebook")

In [ ]:
# @title Prepare startup status, saved references and calibration
# These run inside the workflow's try/finally so Drive is flushed on completion or interruption.
def show_study_progress(inventory, label):
    print(label)
    print(
        "Full runs:", inventory["completed_full_runs"], "/ 12 complete;", inventory["remaining_full_runs"], "remaining"
    )
    print(
        "Training probes:",
        inventory["completed_probe_runs"],
        "/ 4 complete;",
        inventory["remaining_probe_runs"],
        "remaining",
    )
    states = {
        "complete": "Complete",
        "resumable": "Ready to resume",
        "not_started": "Not started",
        "unfinished": "Incomplete — no checkpoint yet",
        "failed": "Needs attention",
    }
    print("Full-run queue:")
    for job in inventory["jobs"]:
        if job["phase"] == "full":
            state = states.get(job["status"], job["status"])
            print("  Arm", job["arm"], "/ seed", str(job["seed"]) + ":", state)
    next_job = inventory.get("next_job")
    if next_job:
        print("Next job:", next_job["name"], "(" + next_job["status"] + ")")
    elif inventory.get("all_jobs_complete"):
        print("Next job: none — all planned jobs are complete.")
    else:
        print("No eligible next job; inspect the recorded errors before continuing.")


def prepare_references_and_calibration():
    from environments.compsognathus.experiments.balance_drive import (
        discover_balance_runs,
        evaluate_saved_baseline,
        save_historical_baselines,
    )
    from environments.compsognathus.experiments.balance_suite import inspect_balance_suite
    from environments.compsognathus.scripts.train_balance_study import prepare_study
    from environments.shared.plant_contract import PlantContractError

    plan = prepare_study(STUDY)
    startup = inspect_balance_suite(STUDY)
    show_study_progress(startup, "Saved progress before reference checks or training:")
    print("Arms:", list(plan["arms"]))
    print("Training seeds:", plan["training_seeds"])
    print("Proposed physical targets:", plan["behavior_targets"])
    print("Package versions:", plan["runtime"])
    snapshot = STUDY / "compsognathus_balance_study.ipynb"
    if not snapshot.exists():
        snapshot.write_bytes((REPO / "notebooks/compsognathus_balance_study.ipynb").read_bytes())

    historical_runs = discover_balance_runs(BASE)
    save_historical_baselines(historical_runs, STUDY / "references" / "historical_baselines.json")
    print("Discovered saved stance runs:", len(historical_runs))
    selected_references = {}
    for species, run_id in {"compsognathus": COMPY_REFERENCE_RUN, "trex": TREX_REFERENCE_RUN}.items():
        matches = [row for row in historical_runs if row["species"] == species and row["run_id"] == run_id]
        if not matches:
            print(
                "Reference not found:",
                species,
                run_id,
                "— available:",
                [row["run_id"] for row in historical_runs if row["species"] == species],
            )
            continue
        selected_references[species] = matches[0]
        print(species, run_id, matches[0]["status"], matches[0].get("saved_metrics", {}))
        print("Saved selected pair:", matches[0]["checkpoint_pair"])

    reference = selected_references.get("compsognathus")
    pair = reference["checkpoint_pair"] if reference else None
    if pair:
        replay_root = STUDY / "references" / "replays" / (COMPY_REFERENCE_RUN + "_n40")
        replay_seeds = tuple(range(11042, 11082))
        # Reuse a completed replay; preserve incomplete attempts and retry them in a new folder.
        completed = [replay_root / "baseline.json"] + sorted(replay_root.glob("attempt_*/baseline.json"))
        baseline = None
        for path in completed:
            if not path.exists():
                continue
            candidate = json.loads(path.read_text())
            if candidate["pair"] != pair:
                raise ValueError("Saved Drive model/config changed since replay; use a new study folder")
            if tuple(row["seed"] for row in candidate["episodes"]) != replay_seeds:
                raise ValueError("Saved baseline replay uses another episode panel")
            baseline = candidate
            print("Reusing completed 40-episode baseline replay:", baseline["summary"])
            break
        if baseline is None:
            replay_root.mkdir(parents=True, exist_ok=True)
            attempt = 1
            while (replay_root / ("attempt_" + str(attempt))).exists():
                attempt += 1
            try:
                baseline = evaluate_saved_baseline(pair, replay_root / ("attempt_" + str(attempt)), seeds=replay_seeds)
                print("Existing Compy policy with new physical measurements:", baseline["summary"])
            except PlantContractError as exc:
                print("Saved model replay blocked by compatibility verification:", str(exc))
                print("The existing reports remain available. No checkpoint identity was changed.")
    else:
        print("Saved reports imported; replay requires an explicitly recorded, available Compy pair.")
    print("Historical control-window support and new physics-level load metrics use different definitions.")

    if not (STUDY / "calibration.json").exists():
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "environments.compsognathus.scripts.train_balance_study",
                "calibrate",
                "--output",
                str(STUDY),
            ]
        )
    calibration = json.loads((STUDY / "calibration.json").read_text())
    home = calibration["home_reference"]
    print("Home reference complete episodes:", sum(row["full_horizon"] for row in home), "/", len(home))

## Run one session and continue next time

Default `session` mode completes any missing probes, then automatically selects one remaining full job. It resumes unfinished work first and validates completed runs before skipping them. If that job completes, this session stops; if the soft time budget expires, training saves a checkpoint and pauses. Run all again with the same study name to resume that job or begin the next one. No manual arm or seed selection is needed. Optional `full` mode continues through all remaining jobs without the session time cap; `smoke` runs only the short checks.

A failed probe prevents full training. Each full job keeps the same 11M-step schedule and four-environment recipe, with screening checkpoints approximately every 250k steps. Screening chooses a model; its exact saved weights and normalization files are then used for the separate 40-seed confirmation panel. A pause checkpoint preserves work without treating an unfinished run as qualified. Off-cadence pause checkpoints are saved separately under `continuations/` and never become checkpoint-selection candidates. Reaching a time limit does not restart or compress the learning-rate or entropy schedule.

Resume preserves policy weights, optimizer state and normalization statistics while resetting simulator episodes. It is not a bit-for-bit continuation of the physical trajectory; unsaved work after an abrupt disconnect may be repeated. The prepared plan checks source and package versions before continuing. Training uses CPU for this small policy, even if Colab provides a GPU.


In [ ]:
# @title Check progress, run this session, then flush Drive
import time

workflow_started = time.monotonic()
try:
    from environments.compsognathus.experiments.balance_suite import inspect_balance_suite, run_balance_suite

    prepare_references_and_calibration()
    remaining_hours = SESSION_HOURS
    if RUN_MODE == "session":
        remaining_hours -= (time.monotonic() - workflow_started) / 3600
    suite = None
    if RUN_MODE == "session" and remaining_hours <= 0:
        print("Session budget used by reference checks/calibration. Training has not started; Run all next session.")
        inventory = inspect_balance_suite(STUDY)
    else:
        if RUN_MODE == "session":
            print("Time remaining in this session:", round(remaining_hours, 2), "hours")
        suite = run_balance_suite(STUDY, mode=RUN_MODE, session_hours=remaining_hours)
        status = suite.get("status")
        if status == "paused":
            reason = suite.get("session_stop_reason")
            if reason == "time_budget_exhausted_before_next_job":
                print("Session time budget reached before starting the next job. Run all next session to continue.")
            elif reason == "trainer_checkpointed_at_time_limit" and any(
                job.get("status") == "paused" and job.get("result", {}).get("checkpoint")
                for job in suite.get("jobs", [])
            ):
                print("Session paused at a saved checkpoint. Run all next session to continue this job.")
            else:
                print("Session paused. Saved progress is shown below; Run all next session to continue.")
        elif status == "session_complete":
            print("This session's full job is complete. Run all next session to start the next remaining job.")
        elif status == "complete":
            print("All jobs requested by this mode are complete.")
        else:
            print("Session status:", status, "— review recorded job errors below.")
        inventory = suite.get("inventory_after") or inspect_balance_suite(STUDY)
    show_study_progress(inventory, "Saved progress at session end:")
    for job in (suite or {}).get("jobs", []):
        if job.get("error"):
            print("Job error:", job["name"], job["error"])
    for name in ("suite_progress.json", "suite_comparison.json", "comparison.json"):
        if (STUDY / name).exists():
            print("Saved report:", STUDY / name)
finally:
    print("Flushing pending Drive writes before this session ends...")
    drive.flush_and_unmount(timeout_ms=300000)
    print("Drive flushed and unmounted. Run all again with the same study name to continue saved work.")

## Read the results

`suite_progress.json` tracks every planned probe and full run, including pauses between sessions. Startup and session-end summaries show completion counts and the next job. `suite_comparison.json` and `comparison.json` include completed, failed and unfinished jobs, so a missing run cannot disappear from the comparison. Existing-model references live under `references/`; `calibration.json` records the home-controller comparison.

Within each full run, `run_summary.json` records whether the selected checkpoint met the proposed behavior targets. `confirmation.json` contains all confirmation episodes, the unchanged stance criteria projected onto canonical reward, and the additional balance criteria. `screening_history.json` records checkpoint selection. `updates.json` records learning rate, exploration coefficient, learned action standard deviation, and optimizer diagnostics. `selected_trace.csv` contains 50 Hz control-boundary traces; physical balance aggregates sample every 2 ms physics step.

Compare physical behavior across reward variants, rather than raw training return. The full-run confirmation panel is separate from training, screening and the workflow-probe panels. A pass here is a research result. Disturbance recovery, deliberate foot unloading, and production promotion remain separate validation steps.
